# 1. Exploratory Data Analysis: LogoSpotter-G[N]

Task 1 (25%). This notebook analyses the group's annotated dataset of television news frames, in which every logo that physically appears in the scene is boxed with the single class `Logo`. It covers the four areas in the specification:

1. **Logo presence**: frames with and without logos, and logo instances per frame.
2. **Logo characteristics**: size distribution, spatial distribution and challenging examples.
3. **Dataset composition**: frames by broadcaster and region, from `frame_metadata.csv`.
4. **Dataset imbalance**: instance distributions and their likely impact on detection.

The dataset is downloaded from Roboflow in COCO format using the shared version in `config.py`. Figures are saved to `figures/` for the technical report.

## Setup

In [ ]:
import json
import re
import sys
from pathlib import Path

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image

sys.path.append(str(Path.cwd()))
from config import FIGURES_DIR, METADATA_CSV, ROBOFLOW_EDA_VERSION, SEED, download_dataset

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(SEED)
FIGURES_DIR.mkdir(exist_ok=True)
SPLITS = ["train", "valid", "test"]


def save(fig, name):
    fig.savefig(FIGURES_DIR / f"eda_{name}.png", dpi=150, bbox_inches="tight")


DATASET = download_dataset("coco", version=ROBOFLOW_EDA_VERSION)
print("Dataset folder:", DATASET)

## Load the annotations

Roboflow renames each frame on export (`img_ab12.jpg` becomes `img_ab12_jpg.rf.<hash>.jpg`) and, when augmentation is enabled, stores several augmented copies of each training frame. The original name is recovered so that each frame is counted once and can be matched to its metadata. Statistics use **one copy per frame**; the number of exported images per split is reported separately.

In [ ]:
def original_name(file_name):
    # Recovers the original frame name from a Roboflow export name.
    name = Path(file_name).name
    m = re.match(r"^(.*)_(jpe?g|png|bmp|webp)\.rf\.[0-9a-f]+\.\w+$", name, re.IGNORECASE)
    return f"{m.group(1)}.{m.group(2)}" if m else name


image_rows, box_rows = [], []
for split in SPLITS:
    ann_file = DATASET / split / "_annotations.coco.json"
    if not ann_file.exists():
        print(f"No {split} split found, skipping")
        continue
    coco = json.loads(ann_file.read_text())
    # Roboflow adds a parent category (supercategory "none"); every other category is a logo.
    logo_ids = {c["id"] for c in coco["categories"] if c.get("supercategory") != "none"}
    logo_ids = logo_ids or {c["id"] for c in coco["categories"]}
    for img in coco["images"]:
        image_rows.append({"split": split, "image_id": img["id"], "file_name": img["file_name"],
                           "frame": original_name(img["file_name"]),
                           "width": img["width"], "height": img["height"]})
    for a in coco["annotations"]:
        if a["category_id"] in logo_ids:
            x, y, w, h = a["bbox"]
            box_rows.append({"split": split, "image_id": a["image_id"], "x": x, "y": y, "w": w, "h": h})

images = pd.DataFrame(image_rows)
boxes = pd.DataFrame(box_rows, columns=["split", "image_id", "x", "y", "w", "h"])
boxes = boxes.merge(images, on=["split", "image_id"], how="left")

exported = images.groupby("split").size().rename("exported images")
# One copy per frame: the first exported image of each frame in each split
first_copy = images.drop_duplicates(["split", "frame"])
frames = first_copy.copy()
boxes = boxes[boxes.set_index(["split", "image_id"]).index.isin(frames.set_index(["split", "image_id"]).index)]

counts = boxes.groupby(["split", "image_id"]).size().rename("logos")
frames = frames.merge(counts, on=["split", "image_id"], how="left").fillna({"logos": 0})
frames["logos"] = frames["logos"].astype(int)
frames["has_logo"] = frames["logos"] > 0

print(f"{len(frames):,} unique frames and {len(boxes):,} logo boxes")
frames.head()

## Frame metadata

`frame_metadata.csv` gives each frame's `video_source`, `timestamp`, `broadcaster` and `region`. It is matched to the annotations by file name.

In [ ]:
metadata = pd.read_csv(METADATA_CSV)
print("Metadata columns:", list(metadata.columns))

# Find the column that holds the frame file names
name_col = next((c for c in ("filename", "file_name", "frame", "image", "image_name", "image_id") if c in metadata.columns),
                None)
if name_col is None:
    name_col = next(c for c in metadata.columns
                    if metadata[c].astype(str).str.contains(r"\.(jpe?g|png)$", case=False).mean() > 0.9)
metadata["frame"] = metadata[name_col].astype(str).map(lambda s: Path(s).name)

frames = frames.merge(metadata.drop(columns=[name_col]), on="frame", how="left")
boxes = boxes.merge(frames[["split", "image_id", "broadcaster", "region"]], on=["split", "image_id"], how="left")
missing = frames["broadcaster"].isna().sum()
print(f"Frames matched to metadata: {len(frames) - missing:,} of {len(frames):,}"
      + (f"  ({missing} unmatched)" if missing else ""))

## 1. Dataset splits

In [ ]:
split_table = frames.groupby("split").agg(
    frames=("frame", "size"),
    frames_with_logos=("has_logo", "sum"),
    null_frames=("has_logo", lambda s: (~s).sum()),
    logo_boxes=("logos", "sum"),
).reindex([s for s in SPLITS if s in frames["split"].unique()])
split_table = split_table.join(exported)
split_table["share of frames"] = (split_table["frames"] / split_table["frames"].sum()).map("{:.1%}".format)
split_table.loc["total"] = split_table.sum(numeric_only=True)
split_table.loc["total", "share of frames"] = "100.0%"
split_table

The `exported images` column is larger than `frames` for the training split when augmentation is applied, because each training frame is exported several times. Augmentation must only be applied to the training set, so `valid` and `test` should have one exported image per frame.

## 2. Logo presence

In [ ]:
n_with, n_without = frames["has_logo"].sum(), (~frames["has_logo"]).sum()
print(f"Frames with logos:    {n_with:,} ({n_with / len(frames):.1%})")
print(f"Frames without logos: {n_without:,} ({n_without / len(frames):.1%})  (null background examples)")
per_frame = frames["logos"]
print(f"Logos per frame: mean {per_frame.mean():.2f}, median {per_frame.median():.0f}, max {per_frame.max()}")
print(f"Logos per frame (frames with logos only): mean {per_frame[per_frame > 0].mean():.2f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
presence = frames.groupby(["split", "has_logo"]).size().unstack(fill_value=0).reindex(
    [s for s in SPLITS if s in frames["split"].unique()])
presence.columns = ["Without logos" if not c else "With logos" for c in presence.columns]
presence.plot(kind="bar", stacked=True, ax=ax1, color=["#bbbbbb", "#3b75af"][: len(presence.columns)])
ax1.set_title("Frames with and without logos, per split")
ax1.set_xlabel("Split")
ax1.set_ylabel("Frames")
ax1.tick_params(axis="x", rotation=0)

max_logos = int(per_frame.max())
ax2.hist(per_frame, bins=np.arange(-0.5, max_logos + 1.5, 1), color="#3b75af", edgecolor="white")
ax2.set_title("Logo instances per frame")
ax2.set_xlabel("Number of logos in the frame")
ax2.set_ylabel("Frames")
fig.tight_layout()
save(fig, "logo_presence")
plt.show()

## 3. Logo characteristics

### Size distribution

Sizes use the COCO convention on the box area in exported pixels: **small** < 32², **medium** 32²–96², **large** ≥ 96². Because frames come from broadcasts of different resolutions, the box area relative to the frame area is also shown.

In [ ]:
boxes["area"] = boxes["w"] * boxes["h"]
boxes["rel_area"] = boxes["area"] / (boxes["width"] * boxes["height"])
boxes["aspect"] = boxes["w"] / boxes["h"].clip(lower=1e-6)
boxes["size"] = pd.cut(boxes["area"], [0, 32 ** 2, 96 ** 2, np.inf], labels=["small", "medium", "large"], right=False)

size_table = boxes["size"].value_counts().reindex(["small", "medium", "large"]).to_frame("boxes")
size_table["share"] = (size_table["boxes"] / size_table["boxes"].sum()).map("{:.1%}".format)
print(size_table, "\n")
print(f"Median box: {boxes['w'].median():.0f} x {boxes['h'].median():.0f} px, "
      f"{boxes['rel_area'].median():.3%} of the frame")
print("Frame resolutions:", ", ".join(f"{w}x{h} ({n})" for (w, h), n in
                                      frames.groupby(["width", "height"]).size().sort_values(ascending=False).items()))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.countplot(data=boxes, x="size", hue="split", order=["small", "medium", "large"], ax=axes[0])
axes[0].set_title("Logo sizes (COCO categories)")
axes[0].set_xlabel("Size")
axes[0].set_ylabel("Boxes")
axes[1].hist(np.log10(boxes["rel_area"].clip(lower=1e-6)), bins=30, color="#3b75af", edgecolor="white")
axes[1].set_title("Box area relative to the frame")
axes[1].set_xlabel("log10(box area / frame area)")
axes[1].set_ylabel("Boxes")
axes[2].hist(np.log2(boxes["aspect"].clip(1 / 32, 32)), bins=30, color="#e07b39", edgecolor="white")
axes[2].set_title("Box aspect ratio")
axes[2].set_xlabel("log2(width / height)   (0 = square, > 0 = wide)")
axes[2].set_ylabel("Boxes")
fig.tight_layout()
save(fig, "logo_sizes")
plt.show()

### Spatial distribution

Box centres are normalised to [0, 1] so frames of different resolutions can be combined. The heatmap shows where in the frame logos tend to appear.

In [ ]:
boxes["cx"] = (boxes["x"] + boxes["w"] / 2) / boxes["width"]
boxes["cy"] = (boxes["y"] + boxes["h"] / 2) / boxes["height"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
h = ax1.hist2d(boxes["cx"], boxes["cy"], bins=[16, 9], range=[[0, 1], [0, 1]], cmap="viridis")
ax1.invert_yaxis()
ax1.set_title("Where logos appear (box centres)")
ax1.set_xlabel("Horizontal position (0 = left, 1 = right)")
ax1.set_ylabel("Vertical position (0 = top, 1 = bottom)")
fig.colorbar(h[3], ax=ax1, label="Boxes")

thirds = pd.crosstab(pd.cut(boxes["cy"], [0, 1 / 3, 2 / 3, 1], labels=["top", "middle", "bottom"], include_lowest=True),
                     pd.cut(boxes["cx"], [0, 1 / 3, 2 / 3, 1], labels=["left", "centre", "right"], include_lowest=True),
                     normalize="all")
sns.heatmap(thirds, annot=True, fmt=".1%", cmap="Blues", cbar=False, ax=ax2)
ax2.set_title("Share of boxes by region of the frame")
ax2.set_xlabel("Horizontal third")
ax2.set_ylabel("Vertical third")
fig.tight_layout()
save(fig, "spatial_distribution")
plt.show()

### Challenging examples

The smallest boxes are the hardest for a detector: there are few pixels to learn from, and at the input resolution of most detectors they shrink further. Below are the smallest logos in the dataset, each shown with some surrounding context.

In [ ]:
def load_frame(row):
    return Image.open(DATASET / row["split"] / row["file_name"]).convert("RGB")


def show_crops(rows, title, fname, pad=2.0):
    fig, axes = plt.subplots(1, len(rows), figsize=(2.4 * len(rows), 2.8))
    for ax, (_, b) in zip(np.atleast_1d(axes), rows.iterrows()):
        img = load_frame(b)
        side = max(b["w"], b["h"]) * (1 + pad)
        cx, cy = b["x"] + b["w"] / 2, b["y"] + b["h"] / 2
        x0, y0 = max(0, cx - side / 2), max(0, cy - side / 2)
        crop = img.crop((x0, y0, min(img.width, x0 + side), min(img.height, y0 + side)))
        ax.imshow(crop)
        ax.add_patch(patches.Rectangle((b["x"] - x0, b["y"] - y0), b["w"], b["h"], fill=False, color="lime", lw=1.5))
        ax.set_title(f"{b['w']:.0f}x{b['h']:.0f} px\n{b.get('broadcaster', '')}", fontsize=8)
        ax.axis("off")
    fig.suptitle(title)
    fig.tight_layout()
    save(fig, fname)
    plt.show()


show_crops(boxes.nsmallest(8, "area"), "Smallest annotated logos (green box), with context", "smallest_logos")

## 4. Dataset composition

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
order_b = frames["broadcaster"].value_counts().index
sns.countplot(data=frames, y="broadcaster", order=order_b, hue="has_logo", ax=axes[0])
axes[0].set_title("Frames per broadcaster")
axes[0].set_xlabel("Frames")
axes[0].set_ylabel("Broadcaster")
axes[0].legend(title="Has logos")

order_r = frames["region"].value_counts().index
sns.countplot(data=frames, x="region", order=order_r, hue="split", ax=axes[1])
axes[1].set_title("Frames per region and split")
axes[1].set_xlabel("Region")
axes[1].set_ylabel("Frames")

per_b = frames.groupby("broadcaster")["logos"].mean().reindex(order_b)
axes[2].barh(per_b.index, per_b.values, color="#3b75af")
axes[2].invert_yaxis()
axes[2].set_title("Average logos per frame, by broadcaster")
axes[2].set_xlabel("Logos per frame")
axes[2].set_ylabel("Broadcaster")
fig.tight_layout()
save(fig, "composition")
plt.show()

composition = frames.groupby(["region", "broadcaster"]).agg(
    frames=("frame", "size"), videos=("video_source", "nunique"),
    logo_boxes=("logos", "sum"), share_with_logos=("has_logo", "mean"))
composition["share_with_logos"] = composition["share_with_logos"].map("{:.1%}".format)
composition

### Split leakage check

Neighbouring frames from the same video can look almost identical. If the same video appears in both the training and test splits, the test score may be optimistic. This counts the videos that appear in more than one split.

In [ ]:
video_splits = frames.groupby("video_source")["split"].nunique()
shared = video_splits[video_splits > 1]
print(f"{len(shared)} of {len(video_splits)} videos appear in more than one split")
if len(shared):
    print(pd.crosstab(frames.loc[frames["video_source"].isin(shared.index), "video_source"],
                      frames.loc[frames["video_source"].isin(shared.index), "split"]).head(10))

## 5. Dataset imbalance

In [ ]:
share_null = (~frames["has_logo"]).mean()
sorted_counts = np.sort(frames.loc[frames["has_logo"], "logos"].to_numpy())[::-1]
top10 = sorted_counts[: max(1, len(sorted_counts) // 10)].sum() / max(1, sorted_counts.sum())
size_share = boxes["size"].value_counts(normalize=True)
box_share_b = boxes["broadcaster"].value_counts(normalize=True)

print(f"Background (null) frames: {share_null:.1%} of all frames")
print(f"The 10% busiest frames hold {top10:.1%} of all logo boxes")
print(f"Small logos: {size_share.get('small', 0):.1%} of boxes, large: {size_share.get('large', 0):.1%}")
print(f"Largest broadcaster share of boxes: {box_share_b.iloc[0]:.1%} ({box_share_b.index[0]})")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
cum = np.cumsum(sorted_counts) / max(1, sorted_counts.sum())
ax1.plot(np.arange(1, len(cum) + 1) / max(1, len(cum)), cum, color="#3b75af")
ax1.plot([0, 1], [0, 1], "--", color="grey", label="Even spread")
ax1.set_title("How concentrated the logo boxes are")
ax1.set_xlabel("Share of frames with logos (busiest first)")
ax1.set_ylabel("Share of all logo boxes")
ax1.legend()
box_share_b.plot(kind="bar", ax=ax2, color="#e07b39")
ax2.set_title("Share of logo boxes by broadcaster")
ax2.set_xlabel("Broadcaster")
ax2.set_ylabel("Share of boxes")
ax2.tick_params(axis="x", rotation=45)
fig.tight_layout()
save(fig, "imbalance")
plt.show()

## 6. Sample annotated frames

In [ ]:
def show_frames(rows, title, fname, cols=3):
    n = len(rows)
    fig, axes = plt.subplots(int(np.ceil(n / cols)), cols, figsize=(5.5 * cols, 3.4 * np.ceil(n / cols)))
    for ax in np.atleast_1d(axes).flat:
        ax.axis("off")
    for ax, (_, f) in zip(np.atleast_1d(axes).flat, rows.iterrows()):
        ax.imshow(load_frame(f))
        for _, b in boxes[(boxes["split"] == f["split"]) & (boxes["image_id"] == f["image_id"])].iterrows():
            ax.add_patch(patches.Rectangle((b["x"], b["y"]), b["w"], b["h"], fill=False, color="lime", lw=1.5))
        ax.set_title(f"{f['split']} | {f.get('broadcaster', '')} | {f['logos']} logo(s)", fontsize=9)
    fig.suptitle(title)
    fig.tight_layout()
    save(fig, fname)
    plt.show()


with_logos = frames[frames["has_logo"]]
show_frames(with_logos.sample(min(6, len(with_logos)), random_state=SEED), "Sample frames with annotated logos",
            "sample_frames")
show_frames(frames.nlargest(3, "logos"), "Frames with the most logos", "busiest_frames")

## 7. Summary

In [ ]:
summary = {
    "unique frames": len(frames),
    "logo boxes": len(boxes),
    "frames with logos": f"{frames['has_logo'].mean():.1%}",
    "logos per frame (mean)": round(frames["logos"].mean(), 2),
    "small / medium / large": " / ".join(f"{size_share.get(s, 0):.1%}" for s in ("small", "medium", "large")),
    "median box (px)": f"{boxes['w'].median():.0f} x {boxes['h'].median():.0f}",
    "broadcasters": frames["broadcaster"].nunique(),
    "regions": frames["region"].nunique(),
    "videos in more than one split": len(shared),
}
pd.Series(summary, name="value").to_frame()

## Key findings and implications for detection

-> _Write these up once the real dataset is in, quoting the numbers above. Points to cover:_

- _**Logo presence**: share of null frames; why background frames help reduce false positives._
- _**Size**: share of small logos; small objects motivate a higher input resolution (e.g. 960–1280 instead of 640) or tiling._
- _**Position**: whether logos cluster in certain regions of the frame; whether flips and crops are safe augmentations._
- _**Composition**: which broadcasters and regions dominate; whether the detector may generalise poorly to under-represented ones._
- _**Imbalance and leakage**: how concentrated the boxes are; whether frames from the same video span splits._